# Entrenamiento CNN U-Net con ResNet50 Transfer Learning (RadImageNet)

## FASES
1. **Phase 1 (25 epochs):** Encoder congelado → Learn decoder only
2. **Phase 2 (10 epochs):** Fine-tuning mejores 30 encoder layers con LR = 1e-5

In [ ]:
# Dependencias
# %pip install numpy tensorflow nibabel matplotlib scikit-image scipy --quiet

In [ ]:
# 0. IMPORTS
import os
import numpy as np
import nibabel as nib
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from scipy import ndimage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✔ All libraries imported")
print(f"   TensorFlow: {tf.__version__}")

## Configuración y carga del dataset

In [ ]:
# 1. CONFIGURACIÓN DE DATASET

DATASET_PATH = "dataset"
IMG_TARGET = 120
BATCH_SIZE = 4

# CONFIGURACIÓN DEL ENTRENAMIENTO
# Opciones:
#   "cerebro"  -> segmentar la máscara de cerebro
#   "isquemia" -> segmentar la máscara de isquemia

TRAIN_TARGET = "cerebro"

# Si es True, la imagen MRI se multiplica por la máscara de cerebro para eliminar fondo (para entrenamiento de isquemia)
APPLY_BRAIN_MASK = False

# Nombre base del modelo guardado
MODEL_NAME = f"{TRAIN_TARGET}_unet_model.h5"

In [ ]:
# 2. CARGA DEL DATASET

# Funciones cargado
def load_nifti_slices(nifti_path):
    """Load NIfTI and normalize to 0-1."""
    nii = nib.load(nifti_path)
    vol = nii.get_fdata()
    vol = (vol - np.min(vol)) / (np.max(vol) - np.min(vol))
    vol = np.rot90(vol, k=3, axes=(0, 1))
    vol = np.flip(vol, axis=1)
    return vol

def load_mask(mask_path):
    """Load PNG mask and convert to binary."""
    img = Image.open(mask_path).convert("L")
    mask = np.array(img, dtype=np.uint8)
    mask = (mask > 127).astype(np.uint8)
    return mask
def load_nifti_slices(nifti_path):
    """Load NIfTI and normalize to 0-1."""
    nii = nib.load(nifti_path)
    vol = nii.get_fdata()

    vol_min, vol_max = np.min(vol), np.max(vol)
    if vol_max > vol_min:
        vol = (vol - vol_min) / (vol_max - vol_min)
    else:
        vol = np.zeros_like(vol, dtype=np.float32)

    vol = np.rot90(vol, k=3, axes=(0, 1))
    vol = np.flip(vol, axis=1)
    return vol

def load_mask(mask_path):
    """Load PNG mask and convert to binary."""
    img = Image.open(mask_path).convert("L")
    mask = np.array(img, dtype=np.uint8)
    mask = (mask > 127).astype(np.uint8)
    return mask

def find_nifti_file(case_path):
    """Busca el archivo NIfTI tolerando diferencias de mayúsculas/minúsculas."""
    candidates = [
        "Seq No.nii",
        "seq NO.nii",
        "seq No.nii",
        "SEQ NO.nii",
    ]

    for fname in candidates:
        full_path = os.path.join(case_path, fname)
        if os.path.exists(full_path):
            return full_path

    for fname in os.listdir(case_path):
        if fname.lower().endswith(".nii") and fname.lower().replace("_", " ") in {
            "seq no.nii",
            "seq no .nii",
        }:
            return os.path.join(case_path, fname)

    return None

def resize_slice_and_mask(img, mask):
    """Resize image and mask to IMG_TARGET."""
    img_p = Image.fromarray((img * 255).astype(np.uint8))
    mask_p = Image.fromarray(mask.astype(np.uint8))

    if img_p.size != (IMG_TARGET, IMG_TARGET):
        img_p = img_p.resize((IMG_TARGET, IMG_TARGET))
    if mask_p.size != (IMG_TARGET, IMG_TARGET):
        mask_p = mask_p.resize((IMG_TARGET, IMG_TARGET), resample=Image.NEAREST)

    img_arr = np.array(img_p, dtype=np.float32) / 255.0
    mask_arr = np.array(mask_p, dtype=np.float32)

    return img_arr[..., np.newaxis], mask_arr[..., np.newaxis]

def get_training_pairs(train_target=TRAIN_TARGET, apply_brain_mask=APPLY_BRAIN_MASK):
    train_target = train_target.lower().strip()
    if train_target not in {"cerebro", "isquemia"}:
        raise ValueError("TRAIN_TARGET debe ser 'cerebro' o 'isquemia'")

    X, Y = [], []
    positive_count = 0
    negative_count = 0

    for case in sorted(os.listdir(DATASET_PATH)):
        case_path = os.path.join(DATASET_PATH, case)
        if not os.path.isdir(case_path):
            continue

        nifti_file = find_nifti_file(case_path)
        if nifti_file is None:
            print(f"⚠️ No se encontró NIfTI en {case_path}")
            continue

        brain_mask_folder = os.path.join(case_path, "cerebro")
        target_mask_folder = os.path.join(case_path, train_target)

        if not os.path.exists(brain_mask_folder):
            print(f"⚠️ No existe carpeta de máscaras de cerebro en {case_path}")
            continue

        print(f"📄 Cargando {nifti_file}")
        vol = load_nifti_slices(nifti_file)
        total_slices = vol.shape[2]

        # Índices con máscara positiva de la tarea objetivo
        target_slices = {}
        if os.path.exists(target_mask_folder):
            for fname in sorted(os.listdir(target_mask_folder)):
                if fname.endswith(".png"):
                    slice_num_str = fname.split("-")[0]
                    slice_idx = int(slice_num_str) - 1
                    target_slices[slice_idx] = fname

        # Recorremos siempre las slices guiándonos por la máscara de cerebro
        for brain_fname in sorted(os.listdir(brain_mask_folder)):
            if not brain_fname.endswith(".png"):
                continue

            slice_num_str = brain_fname.split("-")[0]
            slice_idx = int(slice_num_str) - 1

            if slice_idx < 0 or slice_idx >= total_slices:
                continue

            img = np.squeeze(vol[:, :, slice_idx])
            brain_mask = np.squeeze(load_mask(os.path.join(brain_mask_folder, brain_fname)))

            # Definir máscara objetivo
            if train_target == "cerebro":
                target_mask = brain_mask
                has_target = np.any(target_mask > 0)
            else:
                if slice_idx in target_slices:
                    target_mask = np.squeeze(
                        load_mask(os.path.join(target_mask_folder, target_slices[slice_idx]))
                    )
                    has_target = True
                else:
                    target_mask = np.zeros_like(brain_mask, dtype=np.uint8)
                    has_target = False

            if apply_brain_mask:
                img = img * brain_mask

            img_arr, mask_arr = resize_slice_and_mask(img, target_mask)

            X.append(img_arr)
            Y.append(mask_arr)

            if has_target:
                positive_count += 1
            else:
                negative_count += 1

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    print("✔ Dataset preparado")
    print(f"   Imágenes: {X.shape}")
    print(f"   Máscaras: {Y.shape}")
    print(f"   Slices positivas ({train_target}): {positive_count}")
    print(f"   Slices negativas ({train_target}): {negative_count}")

    return X, Y


In [ ]:
X, Y = get_training_pairs(train_target=TRAIN_TARGET, apply_brain_mask=APPLY_BRAIN_MASK)
print("\n📊 Dataset loaded successfully")


## U-NET y RESNET50

In [ ]:
# 3. LOSS FUNCTIONS

def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


In [ ]:
# 4. U-NET + RESNET50 ENCODER (RADIMAGENET WEIGHTS)

from tensorflow.keras.applications import ResNet50

def conv_block(x, filters):
    """Simple conv block with 2x Conv2D."""
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def unet_model():
    """U-Net with ResNet50 encoder and skip connections."""
    inputs = layers.Input((IMG_TARGET, IMG_TARGET, 1))

    # 1 channel -> 3 channels
    x = layers.Concatenate()([inputs, inputs, inputs])

    # ResNet50 encoder pretrained on RadImageNet
    radimagenet_weights = "models/RadImageNet-ResNet50_notop.h5"
    if not os.path.exists(radimagenet_weights):
        raise FileNotFoundError(f"RadImageNet weights not found: {radimagenet_weights}")

    base_model = ResNet50(
        input_tensor=x,
        include_top=False,
        weights=None
    )
    base_model.load_weights(radimagenet_weights)
    base_model.trainable = False

    # Get skip connections from ResNet50 layers
    s1 = base_model.get_layer("conv1_relu").output          # ~60x60, 64 filters
    s2 = base_model.get_layer("conv2_block3_out").output     # ~30x30, 256 filters
    s3 = base_model.get_layer("conv3_block4_out").output     # ~15x15, 512 filters
    bn = base_model.get_layer("conv4_block6_out").output     # ~8x8, 1024 filters

    # Decoder - Progressive upsampling
    u3 = layers.UpSampling2D()(bn)
    u3 = layers.Resizing(s3.shape[1], s3.shape[2])(u3)
    u3 = layers.Concatenate()([u3, s3])
    c4 = conv_block(u3, 256)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.Resizing(s2.shape[1], s2.shape[2])(u2)
    u2 = layers.Concatenate()([u2, s2])
    c5 = conv_block(u2, 128)

    u1 = layers.UpSampling2D()(c5)
    u1 = layers.Resizing(s1.shape[1], s1.shape[2])(u1)
    u1 = layers.Concatenate()([u1, s1])
    c6 = conv_block(u1, 64)

    u0 = layers.UpSampling2D()(c6)
    u0 = layers.Resizing(IMG_TARGET, IMG_TARGET)(u0)
    c7 = conv_block(u0, 32)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c7)

    model = models.Model(inputs, outputs)
    return model



## Configuración entrenamiento

In [ ]:
# 5. TRAINING SETUP Y CALLBACKS

class DisplayPredictions(tf.keras.callbacks.Callback):
    """Visualize predictions during training."""
    def __init__(self, X_sample, Y_sample):
        self.X_sample = X_sample
        self.Y_sample = Y_sample

    def on_epoch_end(self, epoch, logs=None):
        idx = np.random.randint(len(self.X_sample))
        x = self.X_sample[idx:idx+1]
        y_true = self.Y_sample[idx]
        y_pred = self.model.predict(x, verbose=0)[0, :, :, 0]

        plt.close('all')
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(x[0, :, :, 0], cmap='gray')
        plt.title("Input")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(y_true[:, :, 0], cmap='Greens')
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(x[0, :, :, 0], cmap='gray')
        plt.imshow(y_pred, cmap='Reds', alpha=0.4)
        plt.title(f"Prediction (Epoch {epoch+1})")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

def unfreeze_pretrained_encoder(model, last_n_layers=30):
    """Unfreeze only the last N layers of the ResNet50 encoder for fine-tuning."""
    encoder = None
    for layer in model.layers:
        if isinstance(layer, models.Model) and "resnet50" in layer.name.lower():
            encoder = layer
            break

    if encoder is None:
        print("⚠️  ResNet50 encoder not found")
        return

    encoder.trainable = True
    
    # Freeze all layers except the last last_n_layers
    for layer in encoder.layers[:-last_n_layers]:
        layer.trainable = False

    for layer in encoder.layers[-last_n_layers:]:
        layer.trainable = True

    print(f"✅ Encoder found: {encoder.name}")
    print(f"   Last {last_n_layers} layers unfrozen for fine-tuning")


# Entrenamiento

In [ ]:
# 6. ENTRENAMIENTO (Fase 1 y 2)

model = unet_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=bce_dice_loss,
    metrics=[dice_coef]
)

callback_vis = DisplayPredictions(X[:5], Y[:5])

print("=" * 70)
print("PHASE 1: ENCODER FROZEN (learn decoder only)")
print("=" * 70)
print(f"Training on {len(X)} samples with validation split = 0.15\n")

history_1 = model.fit(
    X, Y,
    batch_size=BATCH_SIZE,
    epochs=25,
    validation_split=0.15,
    verbose=1,
    callbacks=[callback_vis]
)

print("\n" + "=" * 70)
print("PHASE 2: FINE-TUNE ENCODER (top 30 layers)")
print("=" * 70)

unfreeze_pretrained_encoder(model, last_n_layers=30)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=bce_dice_loss,
    metrics=[dice_coef]
)

history_2 = model.fit(
    X, Y,
    batch_size=BATCH_SIZE,
    epochs=10,
    validation_split=0.15,
    verbose=1,
    callbacks=[callback_vis]
)

print("\n✅ Entrenamiento completado!")

## Gráficas

In [ ]:
# 7. CURVAS ENTRENAMIENTO

loss_total = history_1.history['loss'] + history_2.history['loss']
val_loss_total = history_1.history['val_loss'] + history_2.history['val_loss']
dice_total = history_1.history['dice_coef'] + history_2.history['dice_coef']
val_dice_total = history_1.history['val_dice_coef'] + history_2.history['val_dice_coef']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_total, label='loss (train)', linewidth=2)
plt.plot(val_loss_total, label='loss (val)', linewidth=2)
plt.axvline(x=25, color='red', linestyle='--', alpha=0.6, linewidth=2, label='Phase 2 start')
plt.title("Loss - U-Net with ResNet50", fontsize=13, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(dice_total, label='dice_coef (train)', linewidth=2)
plt.plot(val_dice_total, label='dice_coef (val)', linewidth=2)
plt.axvline(x=25, color='red', linestyle='--', alpha=0.6, linewidth=2, label='Phase 2 start')
plt.title("Dice Coefficient - U-Net with ResNet50", fontsize=13, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"   Validation Dice:  {val_dice_total[-1]:.4f}")
print(f"   Validation Loss:  {val_loss_total[-1]:.4f}")
print(f"   Best Dice: {max(val_dice_total):.4f}")

## Guardado

In [ ]:
# 8. GUARDADO DE MODELo

MODEL_DIR = "modelss"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, MODEL_NAME)
model.save(model_path)

print(f"💾 Modelo guardado en: {model_path}")
